In [0]:
from nupack import *

In [0]:
from nupack import *
config.parallelism = True

In [0]:
from nupack import *
config.cache = 8.0 # GB

In [0]:
domains = [Domain('N6', name=['a', i]) for i in range(4)]
print([d.name for d in domains]) # --> ['a[0]', 'a[1]', 'a[2]', 'a[3]']

In [0]:
s = Domain('N6', name='a')
print(str(s)) # --> NNNNNN
print(repr(s)) # --> <Domain a>

In [0]:
# Specify source RNA for window constraints
my_model = Model(material='rna', celsius=37) # set physical parameters
crosstalkTargets = {} # empty crosstalk tube targets
crosstalkExcludes = [] # empty crosstalk tube excludes
tubes = [] # empty set of tubes
systems = 2 #set number of systems

In [0]:
for i in range(systems):

    # define domains
    a = Domain('N6', name=['a', i])
    c = Domain('N8', name=['c', i])
    b = Domain('N4', name=['b', i])
    w = Domain('N2', name=['w', i])
    y = Domain('N4', name=['y', i])
    x = Domain('N12',name=['x', i])
    z = Domain('N3', name=['z', i])
    s = Domain('N5', name=['s', i])

    # define strands from domains
    Cout_s   = TargetStrand([w, x, y, s], name=['Cout_s', i])
    A_s      = TargetStrand([~c, ~b, ~a, ~z, ~y], name=['A_s', i])
    A_toe_s  = TargetStrand([~c], name=['A_toe_s', i])
    C_s      = TargetStrand([w, x, y, s, ~a, ~z, ~y, ~x, ~w], name=['C_s', i])
    C_loop_s = TargetStrand([s, ~a, ~z], name=['C_loop_s', i])
    B_s      = TargetStrand([x, y, z, a, b], name=['B_s', i])
    Xs_s     = TargetStrand([a, b, c], name=['Xs_s', i])

    # define complexes composed of one or more strands in a given order AND
    # define target structures for each complex
    C      = TargetComplex([C_s],       'D2 D12 D4( U5 U6 U3 )', name=['C', i])
    B      = TargetComplex([B_s],       'U12 U4 U3 U6 U4', name=['B', i])
    C_loop = TargetComplex([C_loop_s],  'U14', name=['C_loop', i])
    A_B    = TargetComplex([A_s, B_s],  'U8 D4 D6 D3 D4(+ U12)', name=['A_B', i])
    X      = TargetComplex([Xs_s],      'U18', name=['X', i])
    X_A    = TargetComplex([Xs_s, A_s], 'D6 D4 D8(+) U3 U4', name=['X_A', i])
    C_out  = TargetComplex([Cout_s],    'U23', name=['C_out', i])
    B_C    = TargetComplex([B_s, C_s],  'D12 D4 D3 D6 (U4 + U2 U12 U4 U5) U2', name=['B_C', i])
    A_toe  = TargetComplex([A_toe_s],   'U8', name=['A_toe', i])

    # on-target tubes
    Step_0 = TargetTube({C: 1e-08, X: 1e-08, A_B: 1e-08}, max_size=2, include=[[A_s], [B_s]], exclude=[X_A], name=['Step_0', i])

    Step_1 = TargetTube({X_A: 1e-08, B: 1e-08}, max_size=2, include=[X, A_B], name=['Step_1', i])

    Step_2 = TargetTube({B_C: 1e-08}, max_size=2, include=[B, C], name=['Step_2', i])

    # Crosstalk tube elements
    crosstalkTargets.update({
        A_B: 1e-08,
        C: 1e-08,
        X: 1e-08,
        B: 1e-08,
        C_out: 1e-08,
        C_loop: 1e-08,
        A_toe: 1e-08,
    })

    crosstalkExcludes += [X_A, B_C, [Xs_s, A_toe_s], [B_s, C_loop_s]]

    # Add tubes
    tubes += [Step_0, Step_1, Step_2]

crosstalk = TargetTube(crosstalkTargets, max_size=2, exclude=crosstalkExcludes, name='crosstalk')
tubes.append(crosstalk)

In [0]:
weights = Weights(tubes)
weights[crosstalk] *= systems

In [0]:
my_design = tube_design(tubes, model=my_model, defect_weights=weights)